In [ ]:
import json
import subprocess
from datetime import datetime, timezone
from collections import Counter, defaultdict

import pandas as pd
import ipywidgets as widgets
import networkx as nx

from IPython.display import display, JSON, Markdown, clear_output

NAMESPACE = "web01"
TIMEOUT = 20


def collect_ruleset(namespace=NAMESPACE):
    result = subprocess.run(
        [
            "ip", "netns", "exec", namespace,
            "nft", "-j", "list", "ruleset",
        ],
        capture_output=True,
        text=True,
        timeout=TIMEOUT,
        check=False,
    )

    if result.returncode:
        raise RuntimeError(result.stderr.strip())

    payload = json.loads(result.stdout)

    if not isinstance(payload, dict):
        raise ValueError("Expected a JSON object at the root")

    if not isinstance(payload.get("nftables"), list):
        raise ValueError("Missing nftables object list")

    return {
        "namespace": namespace,
        "collected_at": datetime.now(timezone.utc).isoformat(),
        "ruleset": payload,
    }


snapshot = collect_ruleset()
objects = snapshot["ruleset"]["nftables"]

print("Namespace:", snapshot["namespace"])
print("Collected:", snapshot["collected_at"])
print("Top-level objects:", len(objects))

In [ ]:
idx = defaultdict(list)

for entry in objects:
    if isinstance(entry, dict) and len(entry) == 1:
        kind, value = next(iter(entry.items()))

        if isinstance(value, dict):
            idx[kind].append(value)


map_objects = idx["map"]
set_objects = idx["set"]
rule_objects = idx["rule"]
chain_objects = idx["chain"]
table_objects = idx["table"]


def identity(obj):
    return (
        obj.get("family"),
        obj.get("table"),
        obj.get("name"),
    )


def display_identity(obj):
    family, table, name = identity(obj)
    return f"{family} {table} / {name}"


map_rows = []

for obj in map_objects + set_objects:
    embedded = obj.get("elem", [])

    map_rows.append({
        "family": obj.get("family"),
        "table": obj.get("table"),
        "name": obj.get("name"),
        "kind": "map" if obj in map_objects else "set",
        "key_type": obj.get("type"),
        "value_type": obj.get("map"),
        "flags": obj.get("flags", []),
        "handle": obj.get("handle"),
        "embedded_element_count": (
            len(embedded) if isinstance(embedded, list) else None
        ),
        "raw_element_json": json.dumps(
            embedded, sort_keys=True
        ),
    })


maps_df = pd.DataFrame(map_rows)

print("Maps:", len(map_objects))
print("Sets:", len(set_objects))
print("Rules:", len(rule_objects))
print(
    "Embedded element entries:",
    sum(
        len(obj.get("elem", []))
        for obj in map_objects + set_objects
        if isinstance(obj.get("elem", []), list)
    ),
)

display(
    maps_df.sort_values(
        ["family", "table", "name"]
    ).reset_index(drop=True)
)

In [ ]:
def find_vmap_expressions(value, path="$"):
    found = []

    if isinstance(value, dict):
        for key, child in value.items():
            child_path = f"{path}.{key}"

            if key == "vmap":
                found.append({
                    "path": child_path,
                    "expression": child,
                })

            found.extend(find_vmap_expressions(child, child_path))

    elif isinstance(value, list):
        for i, child in enumerate(value):
            found.extend(
                find_vmap_expressions(child, f"{path}[{i}]")
            )

    return found


vmap_samples = []

for rule in rule_objects:
    matches = find_vmap_expressions(rule.get("expr", []))

    for match in matches:
        vmap_samples.append({
            "family": rule.get("family"),
            "table": rule.get("table"),
            "chain": rule.get("chain"),
            "handle": rule.get("handle"),
            "vmap": match["expression"],
        })


print("Rules with vmap expressions:", len(vmap_samples))

for sample in vmap_samples[:5]:
    display(JSON(sample))

In [ ]:
def parse_rule_map_references(rule):
    rows = []

    for match in find_vmap_expressions(rule.get("expr", [])):
        expression = match["expression"]

        if not isinstance(expression, dict):
            continue

        # nft JSON encodes the vmap target in "data".
        target = expression.get("data")

        if isinstance(target, str):
            target = target.removeprefix("@")

            rows.append({
                "rule_family": rule.get("family"),
                "rule_table": rule.get("table"),
                "rule_chain": rule.get("chain"),
                "rule_handle": rule.get("handle"),
                "map_name": target,
                "lookup_key_json": json.dumps(
                    expression.get("key"),
                    sort_keys=True,
                ),
                "vmap_json": json.dumps(
                    expression,
                    sort_keys=True,
                ),
            })

    return rows


reference_rows = []

for rule in rule_objects:
    reference_rows.extend(parse_rule_map_references(rule))

references_df = pd.DataFrame(reference_rows)

print("Extracted rule-to-map references:", len(references_df))
display(references_df.head(20))

In [ ]:
G = nx.DiGraph()

# Add map and set nodes.
for obj in map_objects + set_objects:
    key = identity(obj)
    node_id = "map:" + ":".join(str(x) for x in key)

    G.add_node(
        node_id,
        kind="map" if obj in map_objects else "set",
        label=f"{key[0]} {key[1]}\n{key[2]}",
        family=key[0],
        table=key[1],
        name=key[2],
    )


def rule_node_id(rule):
    return "rule:" + ":".join(str(x) for x in (
        rule.get("family", "?"),
        rule.get("table", "?"),
        rule.get("chain", "?"),
        rule.get("handle", "?"),
    ))


# Add rule nodes.
for rule in rule_objects:
    G.add_node(
        rule_node_id(rule),
        kind="rule",
        label=(
            f"{rule.get('chain', '?')}\n"
            f"#{rule.get('handle', '?')}"
        ),
        family=rule.get("family"),
        table=rule.get("table"),
        chain=rule.get("chain"),
        handle=rule.get("handle"),
        expressions=json.dumps(rule.get("expr", [])),
    )


# Add directed rule -> map edges.
for row in reference_rows:
    source = "rule:" + ":".join(str(x) for x in (
        row["rule_family"],
        row["rule_table"],
        row["rule_chain"],
        row["rule_handle"],
    ))

    key = (
        row["rule_family"],
        row["rule_table"],
        row["map_name"],
    )
    target = "map:" + ":".join(str(x) for x in key)

    if target not in G:
        G.add_node(
            target,
            kind="unresolved",
            label=f"UNRESOLVED\n{row['map_name']}",
            family=row["rule_family"],
            table=row["rule_table"],
            name=row["map_name"],
        )

    G.add_edge(
        source,
        target,
        relation="vmap",
        lookup_key_json=row["lookup_key_json"],
    )


print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())
print(
    "Resolved edges:",
    sum(
        1 for _, _, d in G.edges(data=True)
        if d.get("relation") == "vmap"
        and G.nodes[_[1]].get("kind") != "unresolved"
    ) if False else sum(
        1 for _, target, d in G.edges(data=True)
        if d.get("relation") == "vmap"
        and G.nodes[target].get("kind") != "unresolved"
    )
)

In [ ]:
vmap_diagnostics = []

for rule in rule_objects:
    for match in find_vmap_expressions(rule.get("expr", [])):
        expression = match["expression"]
        target = (
            expression.get("data")
            if isinstance(expression, dict)
            else None
        )

        vmap_diagnostics.append({
            "family": rule.get("family"),
            "table": rule.get("table"),
            "chain": rule.get("chain"),
            "handle": rule.get("handle"),
            "target_type": type(target).__name__,
            "target": json.dumps(target, sort_keys=True),
            "extracted": isinstance(target, str),
            "expression": json.dumps(
                expression, sort_keys=True
            ),
        })

vmap_diagnostics_df = pd.DataFrame(vmap_diagnostics)

print("Total vmap expressions:", len(vmap_diagnostics_df))
print(
    "String targets:",
    vmap_diagnostics_df["extracted"].sum()
)
print(
    "Non-string or missing targets:",
    (~vmap_diagnostics_df["extracted"]).sum()
)

display(
    vmap_diagnostics_df[
        ~vmap_diagnostics_df["extracted"]
    ]
)

In [ ]:
# Add a derived zone label; preserve the original map name.
maps_df["zone"] = (
    maps_df["name"]
    .astype(str)
    .str.extract(r"^(Z\d+)", expand=False)
    .fillna("Unclassified")
)

zone_options = sorted(maps_df["zone"].unique())

zone_select = widgets.Dropdown(
    options=zone_options,
    description="Zone:",
    value="Z01" if "Z01" in zone_options else zone_options[0],
    layout=widgets.Layout(width="250px"),
)

map_search = widgets.Text(
    description="Filter:",
    placeholder="e.g. tcp4pre",
    layout=widgets.Layout(width="350px"),
)

map_output = widgets.Output()


def show_zone_maps(*_):
    with map_output:
        clear_output(wait=True)

        view = maps_df[
            maps_df["zone"] == zone_select.value
        ].copy()

        query = map_search.value.strip().lower()
        if query:
            view = view[
                view["name"].str.lower().str.contains(
                    query, regex=False
                )
            ]

        columns = [
            "family", "table", "name", "kind",
            "key_type", "value_type", "flags",
            "embedded_element_count", "handle",
        ]

        display(Markdown(
            f"### {zone_select.value}: {len(view)} objects"
        ))
        display(
            view[columns]
            .sort_values(["table", "name"])
            .reset_index(drop=True)
        )


zone_select.observe(show_zone_maps, names="value")
map_search.observe(show_zone_maps, names="value")

display(widgets.VBox([
    widgets.HBox([zone_select, map_search]),
    map_output,
]))

show_zone_maps()

In [ ]:
map_element_rows = []

for obj in map_objects:
    family = obj.get("family")
    table = obj.get("table")
    name = obj.get("name")
    key_type = obj.get("type")
    val_type = obj.get("map")
    
    for entry in obj.get("elem", []):
        # nftables JSON representation can wrap entries as lists/dicts for combined or single keys/values
        if isinstance(entry, list) and len(entry) == 2:
            key_val, target_val = entry
        elif isinstance(entry, dict):
            # Handle specific structural variations if present
            key_val = entry.get("key")
            target_val = entry.get("val")
        else:
            key_val = str(entry)
            target_val = "unknown"

        map_element_rows.append({
            "family": family,
            "table": table,
            "map_name": name,
            "key_type": key_type,
            "value_type": val_type,
            "map_key": json.dumps(key_val),
            "map_value": json.dumps(target_val),
        })

map_elements_df = pd.DataFrame(map_element_rows)

print(f"Total extracted map element pairs: {len(map_elements_df)}")
if not map_elements_df.empty:
    display(map_elements_df.head(15))
else:
    print("No embedded elements found directly in map declarations (elements may be populated dynamically or via control planes).")

In [ ]:
vmap_rows = []

for idx, rule in enumerate(rule_objects):
    matches = find_vmap_expressions(rule.get("expr", []))
    for match in matches:
        expr = match["expression"]
        vmap_rows.append({
            "rule_handle": rule.get("handle"),
            "table": rule.get("table"),
            "chain": rule.get("chain"),
            "family": rule.get("family"),
            "vmap_path": match["path"],
            "vmap_expression_json": json.dumps(expr, sort_keys=True)
        })

vmaps_df = pd.DataFrame(vmap_rows)

print(f"Total vmap expressions found in rules: {len(vmaps_df)}")
if not vmaps_df.empty:
    display(vmaps_df.head(10))

In [ ]:
if map_objects:
    map_options = {display_identity(obj): obj for obj in map_objects}
    
    map_dropdown = widgets.Dropdown(
        options=list(map_options.keys()),
        description='Select Map:',
        style={'description_width': 'initial'}
    )
    
    output_area = widgets.Output()

    def on_map_change(change):
        if change['type'] == 'change' and change['name'] == 'value':
            with output_area:
                clear_output(wait=True)
                selected_obj = map_options[change['new']]
                print("--- Full JSON Definition ---")
                display(JSON(selected_obj))
                
                # Show entries if available
                if "elem" in selected_obj and selected_obj["elem"]:
                    print("\n--- Map Elements ---")
                    elem_df = pd.DataFrame(selected_obj["elem"])
                    display(elem_df)
                else:
                    print("\n(No static elements declared in this map object)")

    map_dropdown.observe(on_map_change)
    
    display(widgets.VBox([map_dropdown, output_area]))
    
    # Trigger initial render
    if map_options:
        on_map_change({'type': 'change', 'name': 'value', 'new': list(map_options.keys())[0]})
else:
    print("No map objects available in the collected ruleset.")

In [ ]:
G = nx.DiGraph()

# Add tables, chains, and maps as nodes with attributes
for obj in table_objects:
    t_name = obj.get("name")
    G.add_node(f"Table: {t_name}", layer="table", type="table")

for obj in chain_objects:
    c_name = obj.get("name")
    t_name = obj.get("table")
    node_id = f"Chain: {t_name}.{c_name}"
    G.add_node(node_id, layer="chain", type="chain")
    G.add_edge(f"Table: {t_name}", node_id)

for obj in map_objects:
    m_name = obj.get("name")
    t_name = obj.get("table")
    node_id = f"Map: {t_name}.{m_name}"
    G.add_node(node_id, layer="map", type="map")
    G.add_edge(f"Table: {t_name}", node_id)

# Connect vmaps in rules to chains/tables if applicable
for _, row in vmaps_df.iterrows():
    r_node = f"Rule #{row['rule_handle']} ({row['chain']})"
    G.add_node(r_node, layer="rule", type="rule")
    G.add_edge(f"Chain: {row['table']}.{row['chain']}", r_node)

print(f"Graph constructed with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.")
print("To render graphically in your notebook environment, you can integrate matplotlib or pyvis if desired.")